In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# 1. Create a dummy image batch: 1 image, 3 channels (RGB), 224x224 pixels
dummy_image = torch.randn(1, 3, 224, 224)
print(f"1. Input image shape: {dummy_image.shape}")

# 2. Load standard ResNet-18 (or ResNet-50)
backbone = models.resnet18(weights=None)  # Using random weights just to inspect shapes
c


# 3. ResNet is divided into stages. Let's trace how the shape evolves:
x = dummy_image
print(f"Input image shape: {x.shape}")
x = backbone.conv1(x)
print(f"After initial Conv1:          {x.shape}  (Downsampled 2x)")
x = backbone.bn1(x) # first Batch Normalization layer
print(f"After initial BatchNorm:      {x.shape}")
x = backbone.relu(x)
print(f"After initial ReLU:           {x.shape}")
x = backbone.maxpool(x)
print(f"After initial Conv1 + MaxPool: {x.shape}  (Downsampled 4x)")

x = backbone.layer1(x)
print(f"3. After Layer 1:                {x.shape}")

x = backbone.layer2(x)
print(f"4. After Layer 2:                {x.shape}  (Downsampled 8x)")

x = backbone.layer3(x)
print(f"5. After Layer 3:                {x.shape}  (Downsampled 16x)")

x = backbone.layer4(x)
print(f"6. After Layer 4:                {x.shape}  (Downsampled 32x)")

# 4. Global Average Pooling (averages each 7x7 slice into 1 number)
x = backbone.avgpool(x)
print(f"7. After Global Avg Pooling:     {x.shape}")

# 5. Flatten to 1D feature vector
feature_vector = torch.flatten(x, 1)
print(f"8. Final 1D Feature Vector:      {feature_vector.shape}")

1. Input image shape: torch.Size([1, 3, 512, 512])
2. After initial Conv1 + MaxPool: torch.Size([1, 64, 128, 128])  (Downsampled 4x)
3. After Layer 1:                torch.Size([1, 64, 128, 128])
4. After Layer 2:                torch.Size([1, 128, 64, 64])  (Downsampled 8x)
5. After Layer 3:                torch.Size([1, 256, 32, 32])  (Downsampled 16x)
6. After Layer 4:                torch.Size([1, 512, 16, 16])  (Downsampled 32x)
7. After Global Avg Pooling:     torch.Size([1, 512, 1, 1])
8. Final 1D Feature Vector:      torch.Size([1, 512])

In [ ]:
# See top-level stages
print("Top-level stages in ResNet:")
for name, module in backbone.named_children():
    print(f" - {name}: {module.__class__.__name__}")

# See inside layer1
print(f"\nInside layer1: {len(backbone.layer1)} BasicBlocks")
print(backbone.layer1[0])  # Inspect the first block

Activity 2: build custom head for 20 class classification

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Step 1: Instantiate the backbone
model = models.resnet18(weights=None)

# Step 2: Check input feature dimension of the original fc layer
in_features = model.fc.in_features
print(f"Original backbone output feature dimension: {in_features}")  # 512 for ResNet-18, 2048 for ResNet-50

# Step 3: Design our custom classification head for 20 classes
num_classes = 20
custom_head = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.3),
    nn.Linear(256, num_classes)
)

print("\n--- FC Head before replacement ---")
print(model.fc)
# Step 4: Replace the default 'fc' with our custom head!
model.fc = custom_head

print("\n--- Our New Model Head ---")
print(model.fc)

# Step 5: Test a forward pass with our dummy image
dummy_image = torch.randn(2, 3, 224, 224)  # Batch of 2 images
logits = model(dummy_image)

print(f"\nBatch input shape:  {dummy_image.shape}")
print(f"Model output shape: {logits.shape}  <-- Exactly (batch_size=2, num_classes=20)!")
print("\nRaw output values (logits) for Image #1:")
print(logits[0].detach().numpy().round(3))

Activity 3: updating loss fn from softmax to BCEWithLogitsLoss

In [ ]:
import torch
import torch.nn as nn

# The 20 assignment classes
CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

# Step 1: Let's create dummy raw logits for 1 image (shape: 1 x 20)
# Suppose the model is confident about 'bicycle' (index 1) and 'person' (index 14)
raw_logits = torch.tensor([[
    -3.2,  3.5, -4.1, -2.8, -3.0,   # aeroplane, bicycle, bird, boat, bottle
    -2.5, -1.8, -4.0, -3.1, -5.0,   # bus, car, cat, chair, cow
    -4.2, -3.8, -4.5, -2.1,  4.2,   # diningtable, dog, horse, motorbike, person
    -3.5, -4.8, -3.9, -4.0, -3.2    # pottedplant, sheep, sofa, train, tvmonitor
]])

# Step 2: Compare Softmax vs Sigmoid
softmax_probs = torch.softmax(raw_logits, dim=-1)
sigmoid_probs = torch.sigmoid(raw_logits)

print("--- Softmax vs Sigmoid ---")
print(f"Softmax sum across all classes: {softmax_probs.sum().item():.2f}  (Forces sum to 1)")
print(f"Sigmoid sum across all classes: {sigmoid_probs.sum().item():.2f}  (Each class independent!)")

print(f"\nProbability of 'bicycle': {sigmoid_probs[0, 1].item():.4f}")
print(f"Probability of 'person':  {sigmoid_probs[0, 14].item():.4f}")
print(f"Probability of 'aeroplane': {sigmoid_probs[0, 0].item():.4f}")

# Step 3: Ground Truth multi-hot label (1 where class is present, 0 elsewhere)
target = torch.zeros(1, 20)
target[0, 1] = 1.0   # bicycle is present
target[0, 14] = 1.0  # person is present

# Step 4: Compute Loss with BCEWithLogitsLoss
criterion = nn.BCEWithLogitsLoss()
loss = criterion(raw_logits, target)
print(f"\nCalculated BCE Loss: {loss.item():.4f}")

# Step 5: Convert probabilities to predictions using threshold >= 0.5 (as per Kaggle rules)
threshold = 0.5
predicted_indices = (sigmoid_probs >= threshold).nonzero(as_tuple=True)[1]
predicted_names = [CLASSES[idx] for idx in predicted_indices]

print(f"\nPredicted Classes (probability >= {threshold}):")
print(" ".join(predicted_names))

Activity 4: aggregating everything

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision import models, transforms
from pathlib import Path

class ClassificationModel(nn.Module):
    def __init__(self, num_classes: int = 20):
        super().__init__()
        
        # 1. Feature Extractor Backbone (ResNet-18)
        self.backbone = models.resnet18(weights=None)
        in_features = self.backbone.fc.in_features  # 512
        
        # 2. Remove default ImageNet head
        self.backbone.fc = nn.Identity()
        
        # 3. Our Custom Multi-Layer Head (from Activity 2)
        self.custom_head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )
        
        # 4. Standard Image Preprocessing for CNNs
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],  # ImageNet standard color mean
                std=[0.229, 0.224, 0.225]    # ImageNet standard color std
            )
        ])
        
        self.classes = [
            'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
            'bus', 'car', 'cat', 'chair', 'cow',
            'diningtable', 'dog', 'horse', 'motorbike', 'person',
            'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
        ]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass for training (returns raw logits)"""
        features = self.backbone(x)
        logits = self.custom_head(features)
        return logits

    @torch.no_grad()
    def predict(self, image: Image.Image, threshold: float = 0.5) -> np.ndarray:
        """Inference method required by assignment (returns binary 0/1 array)"""
        self.eval()
        
        # Preprocess PIL image -> (1, 3, 224, 224)
        img_tensor = self.transform(image.convert("RGB")).unsqueeze(0)
        
        # Forward pass through model
        logits = self.forward(img_tensor)
        
        # Activity 3 logic: Sigmoid + threshold
        probs = torch.sigmoid(logits).squeeze(0)  # Shape: (20,)
        binary_pred = (probs >= threshold).cpu().numpy().astype(np.int64)
        
        return binary_pred


# Test on a real image from the dataset!
sample_image_path = Path("data/Dataset/train/images/img_00001.jpg")
real_img = Image.open(sample_image_path)

model = ClassificationModel()
binary_vector = model.predict(real_img)

print(f"Loaded image: {sample_image_path.name} (Resolution: {real_img.size[0]}x{real_img.size[1]})")
print(f"Output type:  {type(binary_vector)}, Shape: {binary_vector.shape}")
print(f"Binary Vector (20 classes):\n{binary_vector}")

# Print detected class names
detected_classes = [model.classes[i] for i, val in enumerate(binary_vector) if val == 1]
print(f"\nPredicted class names: {detected_classes if detected_classes else 'None (all probabilities < 0.5)'}")

Actual training

In [ ]:
import os
import random
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# -------------------------------------------------------------
# 1. Pure NumPy Macro F1 Score (No sklearn needed!)
# -------------------------------------------------------------
def compute_macro_f1(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-7) -> float:
    tp = np.sum((y_true == 1) & (y_pred == 1), axis=0)
    fp = np.sum((y_true == 0) & (y_pred == 1), axis=0)
    fn = np.sum((y_true == 1) & (y_pred == 0), axis=0)
    
    f1_per_class = (2 * tp) / (2 * tp + fp + fn + eps)
    return float(np.mean(f1_per_class))

# -------------------------------------------------------------
# 2. Device & Seed Configuration
# -------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cpu':
    num_threads = min(16, os.cpu_count() or 4)
    torch.set_num_threads(num_threads)
    print(f"PyTorch CPU threads: {num_threads}")

# Paths
ROOT = Path(".").resolve()
DATA_DIR = Path("data/Dataset") if (ROOT / "data/Dataset").exists() else ROOT / "Dataset"
TRAIN_IMG_DIR = DATA_DIR / "train" / "images"
LABELS_CSV = DATA_DIR / "train" / "labels.csv"

WEIGHTS_DIR = Path("classification/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------
# 3. Classes & Dataset Definition
# -------------------------------------------------------------
CLASSES: List[str] = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

class MultiLabelDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: Path, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.labels = self.df[CLASSES].values.astype(np.float32)
        self.image_ids = self.df['image_id'].values

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img_path = self.img_dir / f"{self.image_ids[idx]}.jpg"
        with Image.open(img_path) as img:
            image = img.convert("RGB")
            
        if self.transform:
            image = self.transform(image)
            
        target = torch.from_numpy(self.labels[idx])
        return image, target

# -------------------------------------------------------------
# 4. Transforms & DataLoaders
# -------------------------------------------------------------
IMAGE_SIZE = 224

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

df_all = pd.read_csv(LABELS_CSV)
val_sample = df_all.sample(frac=0.2, random_state=SEED)
train_sample = df_all.drop(val_sample.index)

train_dataset = MultiLabelDataset(train_sample, TRAIN_IMG_DIR, transform=train_transforms)
val_dataset = MultiLabelDataset(val_sample, TRAIN_IMG_DIR, transform=val_transforms)

BATCH_SIZE = 32 if device.type == 'cuda' else 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Dataset Split: {len(train_dataset)} Train | {len(val_dataset)} Validation")

# -------------------------------------------------------------
# 5. Model Architecture (ResNet-50 + Custom Head)
# -------------------------------------------------------------
class ClassifierNet(nn.Module):
    def __init__(self, num_classes: int = 20):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if hasattr(models, 'ResNet50_Weights') else None)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.head = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(256, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)
        feat = self.pool(feat).flatten(1)
        return self.head(feat)

model = ClassifierNet(num_classes=len(CLASSES)).to(device)

# -------------------------------------------------------------
# 6. Loss, Optimizer & Scheduler
# -------------------------------------------------------------
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
EPOCHS = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# -------------------------------------------------------------
# 7. Training Loop
# -------------------------------------------------------------
best_val_f1 = 0.0
best_model_path = WEIGHTS_DIR / "best_classifier.pth"

print(f"\nStarting training for {EPOCHS} epochs...\n" + "-"*50)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    train_loss /= len(train_dataset)
    scheduler.step()
    
    # Validation
    model.eval()
    val_loss = 0.0
    all_targets, all_preds = [], []
    
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * images.size(0)
            
            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).int()
            
            all_targets.append(targets.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            
    val_loss /= len(val_dataset)
    val_f1 = compute_macro_f1(np.vstack(all_targets), np.vstack(all_preds))
    
    is_best = val_f1 > best_val_f1
    if is_best:
        best_val_f1 = val_f1
        torch.save({
            'state_dict': model.state_dict(),
            'val_f1': val_f1,
            'epoch': epoch
        }, best_model_path)
    
    best_mark = " [BEST SAVED]" if is_best else ""
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Mean F1: {val_f1:.4f}{best_mark}")

print("-" * 50)
print(f"Training Complete! Best Validation Mean F1: {best_val_f1:.4f}")
print(f"Saved weights to: {best_model_path}")

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path
from classification.model import ClassificationModel

# 1. Instantiate using your newly trained weights
clf = ClassificationModel(weights_dir="classification/weights")

# 2. Pick a test image
sample_path = Path("data/Dataset/train/images/img_00003.jpg")
with Image.open(sample_path) as img:
    img_arr = np.array(img.convert("RGB"))

# 3. Predict class probabilities
probs_dict = clf.predict(img_arr)

# 4. Filter classes with probability >= 0.5 (per assignment rules)
predicted_classes = [cls for cls, prob in probs_dict.items() if prob >= 0.5]

print(f"Image: {sample_path.name}")
print(f"Top predicted probabilities:")
sorted_probs = sorted(probs_dict.items(), key=lambda x: x[1], reverse=True)[:5]
for cls, prob in sorted_probs:
    print(f"  - {cls:12s}: {prob:.4f}")

print(f"\nFinal Classification Output: '{' '.join(predicted_classes) if predicted_classes else 'background'}'")

# Segmentation

#### Activity 1: Ground Truth Masks & The "255" Ignore Label

In semantic segmentation benchmarks (like PASCAL VOC), images have object boundaries marked with a special value 255 (void / boundary).

If we treated 255 as a normal class, the model would fail or crash with an index error since our output layer only has 21 channels (
0
0 through 
20
20).
PyTorch provides nn.CrossEntropyLoss(ignore_index=255), which masks out those pixels during gradient calculation.

In [ ]:
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from pathlib import Path

# 1. Inspect a sample training mask
mask_path = Path("./data/Dataset/train/segmentation_masks/img_00032.png")  # or any train mask
if mask_path.exists():
    mask = Image.open(mask_path)
    mask_np = np.array(mask)
    
    unique_vals = np.unique(mask_np)
    print(f"Mask size: {mask.size} (W x H)")
    print(f"Unique pixel values in mask: {unique_vals}")
    print(f"Contains background (0)? {0 in unique_vals}")
    print(f"Contains ignore boundary (255)? {255 in unique_vals}")

Load a sample image and its corresponding segmentation mask from segmentation_masks/.
Inspect the unique pixel values to see the classes present (including the special boundary label 255).
Visualize the raw image, the segmentation mask, and where the boundary/ignore pixels are located.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# 1. Define paths (handles local project structure)
DATA_DIR = Path("data/Dataset/train")
IMG_DIR = DATA_DIR / "images"
MASK_DIR = DATA_DIR / "segmentation_masks"

# The 20 VOC object classes (Class 0 is Background, Classes 1-20 are objects)
CLASS_NAMES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

# 2. Pick a representative sample mask
sample_mask_path = MASK_DIR / "img_00003.png"
sample_img_path = IMG_DIR / "img_00003.jpg"

image = Image.open(sample_img_path).convert("RGB")
mask = Image.open(sample_mask_path)
mask_np = np.array(mask)

# 3. Analyze unique pixel labels
unique_labels = np.unique(mask_np)

print(f"Image Resolution: {image.size[0]} x {image.size[1]} (W x H)")
print(f"Mask Resolution:  {mask_np.shape[1]} x {mask_np.shape[0]} (W x H)")
print(f"Unique pixel values in this mask: {unique_labels}")
print("\nClasses identified:")
for label in unique_labels:
    if label == 0:
        print(f" - Value   0: Background")
    elif label == 255:
        print(f" - Value 255: Boundary / Void (IGNORE LABEL)")
    elif 1 <= label <= 20:
        print(f" - Value {label:3d}: Object -> '{CLASS_NAMES[label]}'")

# 4. Visual Inspection
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot Original Image
axes[0].imshow(image)
axes[0].set_title(f"Original Image: {sample_img_path.name}")
axes[0].axis("off")

# Plot Segmentation Mask (highlighting object labels)
im_mask = axes[1].imshow(mask_np, cmap="tab20", vmin=0, vmax=20)
axes[1].set_title("Ground Truth Mask (Classes 0-20)")
axes[1].axis("off")
fig.colorbar(im_mask, ax=axes[1], fraction=0.046, pad=0.04)

# Plot Ignore Mask (value == 255)
axes[2].imshow(mask_np == 255, cmap="Reds")
axes[2].set_title("Boundary Ignore Pixels (Value = 255)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Paths to sample image and its ground truth mask
img_path = Path("data/Dataset/train/images/img_00001.jpg")
mask_path = Path("data/Dataset/train/segmentation_masks/img_00001.png")

# 1. Load the original image and mask
img = Image.open(img_path)
mask = Image.open(mask_path)

img_arr = np.array(img)
mask_arr = np.array(mask)

print(f"Image Resolution: {img_arr.shape}  (Height, Width, 3 RGB channels)")
print(f"Mask Resolution:  {mask_arr.shape}  (Height, Width) - 2D integer matrix!")

# 2. Inspect unique pixel values in this mask
unique_vals = np.unique(mask_arr)
print(f"\nUnique values present in mask: {unique_vals}")

CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

for val in unique_vals:
    if val == 255:
        print(f" - Value {val:3d}: Ignore boundary (contour pixels)")
    elif val < len(CLASSES):
        print(f" - Value {val:3d}: Class '{CLASSES[val]}'")

# 3. Visualize Image side-by-side with Mask
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_arr)
axes[0].set_title(f"Original: {img_path.name}")
axes[0].axis("off")

# # Use a colormap with discrete colors to see the objects
# axes[1].imshow(mask_arr, cmap="tab20")
# axes[1].set_title(f"Ground Truth Mask: {mask_path.name}")
# axes[1].axis("off")

# Plot Segmentation Mask (highlighting object labels)
im_mask = axes[1].imshow(mask_arr, cmap="tab20", vmin=0, vmax=20)
axes[1].set_title("Ground Truth Mask (Classes 0-20)")
axes[1].axis("off")
fig.colorbar(im_mask, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# -------------------------------------------------------------
# 1. Building Block: Conv + BatchNorm + ReLU
# -------------------------------------------------------------
class ConvBNAct(nn.Module):
    """Standard Convolution + BatchNorm + ReLU block."""
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int | None = None):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


# -------------------------------------------------------------
# 2. ResNet Encoder (Captures intermediate multi-scale feature maps)
# -------------------------------------------------------------
class ResNetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        # Use unweighted model for shape tracing
        backbone = models.resnet50(weights=None)
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1  # 256 channels, 1/4 resolution
        self.layer2 = backbone.layer2  # 512 channels, 1/8 resolution
        self.layer3 = backbone.layer3  # 1024 channels, 1/16 resolution
        self.layer4 = backbone.layer4  # 2048 channels, 1/32 resolution

    def forward(self, x: torch.Tensor):
        x = self.stem(x)
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c1, c2, c3, c4


# -------------------------------------------------------------
# 3. Custom Decoder with Progressive Skip Connections
# -------------------------------------------------------------
class SegmentationDecoder(nn.Module):
    def __init__(self, num_classes: int = 21):
        super().__init__()
        # Reduce 2048 bottleneck channels to 256
        self.proj4 = ConvBNAct(2048, 256, kernel_size=1, padding=0)
        
        # Fuse with Layer 3 (c3: 1024 channels) -> 256 + 1024 = 1280 channels
        self.fuse34 = nn.Sequential(
            ConvBNAct(256 + 1024, 256),
            ConvBNAct(256, 256),
        )
        
        # Fuse with Layer 2 (c2: 512 channels) -> 256 + 512 = 768 channels
        self.fuse23 = nn.Sequential(
            ConvBNAct(256 + 512, 192),
            ConvBNAct(192, 192),
        )
        
        # Fuse with Layer 1 (c1: 256 channels) -> 192 + 256 = 448 channels
        self.fuse12 = nn.Sequential(
            ConvBNAct(192 + 256, 128),
            ConvBNAct(128, 128),
        )
        
        # Final 1x1 classifier conv to project 128 features -> 21 class logits
        self.classifier = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, c1, c2, c3, c4, out_size):
        print(f"--- Decoder Tracing ---")
        # Step 1: Project Layer 4 bottleneck
        x = self.proj4(c4)
        print(f"1. Bottleneck projected:          {x.shape}")

        # Step 2: Upsample 2x and fuse with Layer 3
        x = F.interpolate(x, size=c3.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse34(torch.cat([x, c3], dim=1))
        print(f"2. After fusing with Layer 3:     {x.shape}")

        # Step 3: Upsample 2x and fuse with Layer 2
        x = F.interpolate(x, size=c2.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse23(torch.cat([x, c2], dim=1))
        print(f"3. After fusing with Layer 2:     {x.shape}")

        # Step 4: Upsample 2x and fuse with Layer 1
        x = F.interpolate(x, size=c1.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse12(torch.cat([x, c1], dim=1))
        print(f"4. After fusing with Layer 1:     {x.shape}")

        # Step 5: Final upsample to original image size and 1x1 conv
        x = F.interpolate(x, size=out_size, mode='bilinear', align_corners=False)
        logits = self.classifier(x)
        print(f"5. Final Logits Output:           {logits.shape}  <-- Exactly (B, 21, H, W)!")
        return logits


# -------------------------------------------------------------
# 4. End-to-End Test Run with Dummy Image
# -------------------------------------------------------------
encoder = ResNetBackbone()
decoder = SegmentationDecoder(num_classes=21)

dummy_img = torch.randn(1, 3, 384, 384)
print(f"Input image shape: {dummy_img.shape}\n" + "="*50)

# 1. Forward through Encoder
c1, c2, c3, c4 = encoder(dummy_img)
print("--- Encoder Outputs ---")
print(f"Layer 1 (c1) shape: {c1.shape}  (Downsampled  4x)")
print(f"Layer 2 (c2) shape: {c2.shape}  (Downsampled  8x)")
print(f"Layer 3 (c3) shape: {c3.shape}  (Downsampled 16x)")
print(f"Layer 4 (c4) shape: {c4.shape}  (Downsampled 32x)\n" + "="*50)

# 2. Forward through Decoder
logits = decoder(c1, c2, c3, c4, out_size=dummy_img.shape[-2:])
print("="*50)
print(f"Verification: {logits.shape == (1, 21, 384, 384)}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class MiniSegmentationModel(nn.Module):
    def __init__(self, num_classes: int = 21):
        super().__init__()
        # 1. ENCODER: Pretrained ResNet Backbone
        backbone = models.resnet18(weights=None)
        
        # Save stages to capture skip connections
        self.initial = nn.Sequential(
            backbone.conv1, 
            backbone.bn1, 
            backbone.relu, 
            backbone.maxpool
        )  # Downsamples 4x: (64, 64, 64)
        
        self.layer1 = backbone.layer1  # (64, 64, 64)   - Skip 1
        self.layer2 = backbone.layer2  # (128, 32, 32)  - Skip 2
        self.layer3 = backbone.layer3  # (256, 16, 16)  - Skip 3
        self.layer4 = backbone.layer4  # (512, 8, 8)    - Bottleneck
        
        # 2. DECODER: Upsampling blocks
        # Upsample 8x8 -> 16x16 and combine with Skip 3
        self.up3 = nn.Sequential(
            nn.Conv2d(512 + 256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        
        # Upsample 16x16 -> 32x32 and combine with Skip 2
        self.up2 = nn.Sequential(
            nn.Conv2d(256 + 128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        
        # Upsample 32x32 -> 64x64 and combine with Skip 1
        self.up1 = nn.Sequential(
            nn.Conv2d(128 + 64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        # Final projection to 21 classes and upsample to 256x256
        self.classifier = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        H, W = x.shape[2], x.shape[3]
        
        # --- ENCODER (Downsampling) ---
        x0 = self.initial(x)       # (64, 64, 64)
        x1 = self.layer1(x0)       # (64, 64, 64)  <-- Skip 1
        x2 = self.layer2(x1)       # (128, 32, 32) <-- Skip 2
        x3 = self.layer3(x2)       # (256, 16, 16) <-- Skip 3
        x4 = self.layer4(x3)       # (512, 8, 8)   <-- Bottleneck
        
        # --- DECODER (Upsampling + Skip Connections) ---
        d3 = F.interpolate(x4, size=x3.shape[2:], mode='bilinear', align_corners=False)
        d3 = self.up3(torch.cat([d3, x3], dim=1))
        
        d2 = F.interpolate(d3, size=x2.shape[2:], mode='bilinear', align_corners=False)
        d2 = self.up2(torch.cat([d2, x2], dim=1))
        
        d1 = F.interpolate(d2, size=x1.shape[2:], mode='bilinear', align_corners=False)
        d1 = self.up1(torch.cat([d1, x1], dim=1))
        
        # Final logits
        logits = self.classifier(d1)
        # Restore full original image resolution
        logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
        return logits

# Test with dummy batch of 1 image
dummy_img = torch.randn(1, 3, 256, 256)
seg_model = MiniSegmentationModel(num_classes=21)
output_logits = seg_model(dummy_img)

print(f"Input image shape:   {dummy_img.shape}")
print(f"Output logits shape: {output_logits.shape}  <-- (batch=1, classes=21, H=256, W=256)")

# To get the final predicted mask (integer per pixel), take the argmax across channels:
predicted_mask = torch.argmax(output_logits, dim=1).squeeze(0)
print(f"Predicted 2D mask:   {predicted_mask.shape}  <-- Matrix of class integers!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceLoss(nn.Module):
    """
    Computes Soft Dice Loss for multi-class segmentation.
    Directly maximizes spatial intersection over union.
    """
    def __init__(self, num_classes: int = 21, ignore_index: int = 255, smooth: float = 1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits: (B, C, H, W) -> convert to probabilities via softmax
        probs = F.softmax(logits, dim=1)
        
        # Mask out ignore_index (255)
        valid_mask = (targets != self.ignore_index)
        targets_clean = torch.where(valid_mask, targets, torch.zeros_like(targets))
        
        # Convert targets to one-hot: (B, H, W) -> (B, C, H, W)
        targets_one_hot = F.one_hot(targets_clean, num_classes=self.num_classes).permute(0, 3, 1, 2).float()
        
        # Zero out ignore pixels in both prediction and target
        valid_mask_expanded = valid_mask.unsqueeze(1).float()
        probs = probs * valid_mask_expanded
        targets_one_hot = targets_one_hot * valid_mask_expanded
        
        # Compute Dice per class: 2 * |A n B| / (|A| + |B|)
        intersection = torch.sum(probs * targets_one_hot, dim=(0, 2, 3))
        cardinality = torch.sum(probs + targets_one_hot, dim=(0, 2, 3))
        
        dice_score = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        # Dice Loss = 1 - Dice Score (averaged over classes)
        return 1.0 - torch.mean(dice_score)


# --- Test Composite Loss on Dummy Output and Target ---
B, C, H, W = 2, 21, 64, 64
dummy_logits = torch.randn(B, C, H, W, requires_grad=True)

# Create dummy ground-truth mask with classes 0 (background), 12 (dog), and 255 (boundary)
dummy_target = torch.zeros(B, H, W, dtype=torch.long)
dummy_target[:, 20:40, 20:40] = 12    # Square of class 12
dummy_target[:, 19:21, 19:41] = 255   # Boundary pixels around the object

# 1. Cross-Entropy Loss (with ignore_index=255)
ce_loss_fn = nn.CrossEntropyLoss(ignore_index=255)
ce_loss = ce_loss_fn(dummy_logits, dummy_target)

# 2. Dice Loss
dice_loss_fn = DiceLoss(num_classes=21, ignore_index=255)
dice_loss = dice_loss_fn(dummy_logits, dummy_target)

# 3. Composite Total Loss
total_loss = 0.5 * ce_loss + 0.5 * dice_loss

print(f"Cross-Entropy Loss: {ce_loss.item():.4f}")
print(f"Dice Loss:          {dice_loss.item():.4f}")
print(f"Composite Total:    {total_loss.item():.4f}")

In [ ]:
import numpy as np

# ---------------------------------------------------------
# 1. Compute Mean IoU between Predicted Mask and Ground Truth
# ---------------------------------------------------------
def compute_image_miou(pred_mask: np.ndarray, target_mask: np.ndarray, num_classes: int = 21, ignore_index: int = 255) -> float:
    """
    Computes Mean IoU for a single image, ignoring boundary pixels (255).
    """
    valid = (target_mask != ignore_index)
    p = pred_mask[valid]
    t = target_mask[valid]

    ious = []
    # Check classes present in ground truth or prediction
    present_classes = np.unique(np.concatenate([p, t]))
    
    for c in present_classes:
        intersection = np.sum((p == c) & (t == c))
        union = np.sum((p == c) | (t == c))
        if union == 0:
            continue
        ious.append(intersection / union)

    return float(np.mean(ious)) if ious else 1.0


# ---------------------------------------------------------
# 2. Official Kaggle RLE Encoder & Decoder (from data/README.md)
# ---------------------------------------------------------
def mask_to_rle(mask: np.ndarray) -> str:
    """Convert 2D segmentation mask to RLE string."""
    flat = mask.flatten()  # row-major order
    starts, lengths, values = [], [], []
    i = 0
    while i < len(flat):
        if flat[i] != 0:
            start = i
            val = flat[i]
            while i < len(flat) and flat[i] == val:
                i += 1
            starts.append(start)
            lengths.append(i - start)
            values.append(val)
        else:
            i += 1
            
    parts = []
    for s, l, v in zip(starts, lengths, values):
        parts.extend([str(s), str(l), str(v)])
    return " ".join(parts) if parts else "0 0 0"


def rle_to_mask(rle_string: str, height: int, width: int) -> np.ndarray:
    """Decode RLE string back to a 2D segmentation mask."""
    mask = np.zeros(height * width, dtype=np.uint8)
    if rle_string == "0 0 0" or not rle_string.strip():
        return mask.reshape(height, width)
        
    parts = list(map(int, rle_string.split()))
    for i in range(0, len(parts), 3):
        start, length, value = parts[i], parts[i+1], parts[i+2]
        mask[start : start + length] = value
    return mask.reshape(height, width)


# ---------------------------------------------------------
# 3. Test on the real Dog Mask from Activity 1!
# ---------------------------------------------------------
# Let's simulate a prediction that is slightly imperfect
simulated_pred = mask_arr.copy()
# Replace ignore boundary with background in prediction
simulated_pred[simulated_pred == 255] = 0
# Introduce a small artificial error (blurring 100 pixels)
simulated_pred[150:160, 150:160] = 0

# Test mIoU
score = compute_image_miou(simulated_pred, mask_arr)
print(f"Simulated Prediction mIoU: {score * 100:.2f}%")

# Test RLE Encoding
rle_encoded = mask_to_rle(simulated_pred)
print(f"\nOriginal mask size in memory: {simulated_pred.nbytes} bytes")
print(f"RLE string character length:  {len(rle_encoded)} chars")
print(f"RLE string sample (first 60 chars): {rle_encoded[:60]}...")

# Test RLE Decoding (Verify 100% lossless compression!)
decoded_mask = rle_to_mask(rle_encoded, height=mask_arr.shape[0], width=mask_arr.shape[1])
is_identical = np.array_equal(simulated_pred, decoded_mask)
print(f"\nRLE Round-trip lossless check: {is_identical} (Exact match!)")

Actual Segmentation

In [ ]:
import os
import random
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# -------------------------------------------------------------
# 1. Device & Reproducibility
# -------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cpu':
    num_threads = min(16, os.cpu_count() or 4)
    torch.set_num_threads(num_threads)
    print(f"Configured PyTorch CPU threads: {num_threads}")

# Paths
ROOT = Path(".").resolve()
DATA_DIR = Path("data/Dataset") if (ROOT / "data/Dataset").exists() else ROOT / "Dataset"
TRAIN_IMG_DIR = DATA_DIR / "train" / "images"
TRAIN_MASK_DIR = DATA_DIR / "train" / "segmentation_masks"

WEIGHTS_DIR = Path("segmentation/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------
# 2. Segmentation Dataset with Nearest-Neighbor Masks
# -------------------------------------------------------------
NUM_CLASSES = 21  # 20 objects + 1 background (0)
IMAGE_SIZE = 224   # Efficient spatial resolution for fast convergence

class SegmentationDataset(Dataset):
    def __init__(self, image_paths: List[Path], mask_dir: Path, img_size: int = IMAGE_SIZE, is_train: bool = True):
        self.image_paths = image_paths
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.is_train = is_train
        
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img_p = self.image_paths[idx]
        mask_p = self.mask_dir / f"{img_p.stem}.png"
        
        # Load image & mask
        img = Image.open(img_p).convert("RGB")
        mask = Image.open(mask_p)
        
        # Resize: Bilinear for image, NEAREST for discrete class mask!
        img = img.resize((self.img_size, self.img_size), Image.Resampling.BILINEAR)
        mask = mask.resize((self.img_size, self.img_size), Image.Resampling.NEAREST)
        
        # Random horizontal flip during training
        if self.is_train and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            
        # Convert to tensors
        img_tensor = transforms.ToTensor()(img)
        img_tensor = self.normalize(img_tensor)
        
        mask_tensor = torch.from_numpy(np.array(mask, dtype=np.int64))
        return img_tensor, mask_tensor

# Train/Val Split (80% Train, 20% Val)
all_images = sorted(TRAIN_IMG_DIR.glob("*.jpg"))
random.Random(SEED).shuffle(all_images)

split_idx = int(0.8 * len(all_images))
train_imgs = all_images[:split_idx]
val_imgs = all_images[split_idx:]

train_dataset = SegmentationDataset(train_imgs, TRAIN_MASK_DIR, is_train=True)
val_dataset = SegmentationDataset(val_imgs, TRAIN_MASK_DIR, is_train=False)

BATCH_SIZE = 16 if device.type == 'cuda' else 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Segmentation Split: {len(train_dataset)} Train | {len(val_dataset)} Validation")

# -------------------------------------------------------------
# 3. Model Architecture: ResNet-50 + Multi-Scale Skip Decoder
# -------------------------------------------------------------
class ConvBNAct(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, padding=kernel_size // 2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class CustomSegmentationNet(nn.Module):
    def __init__(self, num_classes: int = 21):
        super().__init__()
        # Backbone (ResNet-50)
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if hasattr(models, 'ResNet50_Weights') else None)
        self.stem = nn.Sequential(
            backbone.conv1, 
            backbone.bn1, 
            backbone.relu, 
            backbone.maxpool
        )
        self.layer1 = backbone.layer1  # 256 channels (c1)
        self.layer2 = backbone.layer2  # 512 channels (c2)
        self.layer3 = backbone.layer3  # 1024 channels (c3)
        self.layer4 = backbone.layer4  # 2048 channels (c4)
        
        # Decoder Fusing Stages
        self.proj4 = ConvBNAct(2048, 256, kernel_size=1)
        self.fuse34 = nn.Sequential(ConvBNAct(256 + 1024, 256), ConvBNAct(256, 256))
        self.fuse23 = nn.Sequential(ConvBNAct(256 + 512, 192), ConvBNAct(192, 192))
        self.fuse12 = nn.Sequential(ConvBNAct(192 + 256, 128), ConvBNAct(128, 128))
        self.classifier = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        H, W = x.shape[2], x.shape[3]
        # Encoder
        x_stem = self.stem(x)
        c1 = self.layer1(x_stem)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        
        # Decoder with Skip Connections
        d = self.proj4(c4)
        d = F.interpolate(d, size=c3.shape[2:], mode='bilinear', align_corners=False)
        d = self.fuse34(torch.cat([d, c3], dim=1))
        
        d = F.interpolate(d, size=c2.shape[2:], mode='bilinear', align_corners=False)
        d = self.fuse23(torch.cat([d, c2], dim=1))
        
        d = F.interpolate(d, size=c1.shape[2:], mode='bilinear', align_corners=False)
        d = self.fuse12(torch.cat([d, c1], dim=1))
        
        # Final logits resized back to input resolution
        logits = self.classifier(d)
        logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
        return logits

model = CustomSegmentationNet(num_classes=NUM_CLASSES).to(device)

# -------------------------------------------------------------
# 4. Composite Loss: Cross-Entropy + Soft Dice Loss
# -------------------------------------------------------------
class DiceLoss(nn.Module):
    def __init__(self, num_classes: int = 21, ignore_index: int = 255, smooth: float = 1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)
        valid_mask = (targets != self.ignore_index)
        targets_clean = torch.where(valid_mask, targets, torch.zeros_like(targets))
        targets_one_hot = F.one_hot(targets_clean, num_classes=self.num_classes).permute(0, 3, 1, 2).float()
        
        valid_mask_exp = valid_mask.unsqueeze(1).float()
        probs = probs * valid_mask_exp
        targets_one_hot = targets_one_hot * valid_mask_exp
        
        intersection = torch.sum(probs * targets_one_hot, dim=(0, 2, 3))
        cardinality = torch.sum(probs + targets_one_hot, dim=(0, 2, 3))
        
        dice_score = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - torch.mean(dice_score)

criterion_ce = nn.CrossEntropyLoss(ignore_index=255)
criterion_dice = DiceLoss(num_classes=NUM_CLASSES, ignore_index=255)

def composite_loss(logits, targets):
    return 0.5 * criterion_ce(logits, targets) + 0.5 * criterion_dice(logits, targets)

# -------------------------------------------------------------
# 5. Optimizer, Scheduler & Evaluation Metric (mIoU)
# -------------------------------------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
EPOCHS = 12
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

def compute_batch_miou(preds: np.ndarray, targets: np.ndarray, ignore_index: int = 255) -> float:
    batch_ious = []
    for p, t in zip(preds, targets):
        valid = (t != ignore_index)
        p_val, t_val = p[valid], t[valid]
        present = np.unique(np.concatenate([p_val, t_val]))
        img_ious = []
        for c in present:
            inter = np.sum((p_val == c) & (t_val == c))
            union = np.sum((p_val == c) | (t_val == c))
            if union > 0:
                img_ious.append(inter / union)
        if img_ious:
            batch_ious.append(np.mean(img_ious))
    return float(np.mean(batch_ious)) if batch_ious else 1.0

# -------------------------------------------------------------
# 6. Training & Validation Loop
# -------------------------------------------------------------
best_val_miou = 0.0
best_model_path = WEIGHTS_DIR / "best_segmentation.pth"

print(f"\nStarting Segmentation training for {EPOCHS} epochs...\n" + "-"*55)

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        logits = model(images)
        loss = composite_loss(logits, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    train_loss /= len(train_dataset)
    scheduler.step()
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_miou_list = []
    
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            logits = model(images)
            loss = composite_loss(logits, masks)
            val_loss += loss.item() * images.size(0)
            
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            targets_np = masks.cpu().numpy()
            val_miou_list.append(compute_batch_miou(preds, targets_np))
            
    val_loss /= len(val_dataset)
    val_miou = float(np.mean(val_miou_list))
    
    is_best = val_miou > best_val_miou
    if is_best:
        best_val_miou = val_miou
        torch.save({
            'state_dict': model.state_dict(),
            'val_miou': val_miou,
            'epoch': epoch
        }, best_model_path)
        
    best_mark = " [BEST SAVED]" if is_best else ""
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val mIoU: {val_miou*100:.2f}%{best_mark}")

print("-" * 55)
print(f"Training Complete! Best Validation mIoU: {best_val_miou*100:.2f}%")
print(f"Saved weights to: {best_model_path}")